In [2]:
#from playwright.sync_api import sync_playwright
from playwright.async_api import async_playwright, expect, TimeoutError
import json
from pathlib import Path
import time
from datetime import date, timedelta
import os

In [3]:
#MY_STATES = ['Colorado', 'Oklahoma', 'Utah', 'California', 'Nevada', "New_Mexico", "Arizona", "North_California"]
MY_STATES = ['700k']  # proxy for all 49 states/districts
# PINNACLE_URLS = "/home/vince/Documents/SmartFires/osm_fitness/Azira/get-urls/urls.txt"
PINNACLE_URLS = "/mnt/beegfs/hellgate/home/vc149353/osm_fitness/Azira/get-urls/urls.txt"

In [26]:
def get_geojson_files(MY_STATES, astuple=False):
    geojson_files = []

    #base_dir = Path('/home/vince/Documents/SmartFires/osm_fitness')
    base_dir = Path('/mnt/beegfs/hellgate/home/vc149353/osm_fitness')
    folders = [p for p in base_dir.iterdir() if p.is_dir() and (sum([s in p.name for s in MY_STATES]) > 0)]

    for folder in folders:  
        curr_files = list(folder.glob('*.geojson'))
        for json_path in curr_files:
            geojson_files.append(("_".join(folder.name.split("_")[:-1]), json_path))  
    
    return [(state, normalize_name(file_path.stem), file_path) for state, file_path in geojson_files]    

def get_not_submitted(urls_file, states):
    geojson_files = get_geojson_files(states)
    with Path(urls_file).open() as f:
        urls =f.read()

    not_downloaded = [f for f in geojson_files if f[1] not in urls]

    # remove parent items ('Sandy' if "Sandy_a" exists)
    results = [f for f in not_downloaded if not has_children(f[1], not_downloaded)]

    return results

def normalize_name(name):
    return name.replace(' ','_').replace(".", "_").replace("(", "_").replace(")","_").replace("'","-").replace("West_Fargo","WestFargo")

def has_children(name, files_list):
    prefix = name + "_a"
    return any(n[0].startswith(prefix) for n in files_list)


In [27]:
len(get_geojson_files(MY_STATES, astuple=True))

1732

In [28]:
not_download_list = get_not_submitted(PINNACLE_URLS, MY_STATES)
not_download_list

[('Washington_DC',
  'Washington_3',
  PosixPath('/mnt/beegfs/hellgate/home/vc149353/osm_fitness/Washington_DC_700k(3)/Washington_3.geojson')),
 ('Washington',
  'Spokane_1',
  PosixPath('/mnt/beegfs/hellgate/home/vc149353/osm_fitness/Washington_700k(50)/Spokane_1.geojson')),
 ('Washington',
  'Kennewick',
  PosixPath('/mnt/beegfs/hellgate/home/vc149353/osm_fitness/Washington_700k(50)/Kennewick.geojson')),
 ('Wyoming',
  'Cheyenne',
  PosixPath('/mnt/beegfs/hellgate/home/vc149353/osm_fitness/Wyoming_700k(6)/Cheyenne.geojson')),
 ('Wyoming',
  'Riverton',
  PosixPath('/mnt/beegfs/hellgate/home/vc149353/osm_fitness/Wyoming_700k(6)/Riverton.geojson')),
 ('Wyoming',
  'Gillette',
  PosixPath('/mnt/beegfs/hellgate/home/vc149353/osm_fitness/Wyoming_700k(6)/Gillette.geojson')),
 ('Wyoming',
  'Laramie',
  PosixPath('/mnt/beegfs/hellgate/home/vc149353/osm_fitness/Wyoming_700k(6)/Laramie.geojson')),
 ('Wyoming',
  'Rock_Springs',
  PosixPath('/mnt/beegfs/hellgate/home/vc149353/osm_fitness/Wyomi

## Write not downloaded list

In [30]:
with open("not_downloaded_012226.txt", "w") as f:
    for state, name, path in not_download_list: 
        f.write(f"{path}\n")